# 04 -- Session and news analysis

Paired scripts: `analysis/join_news_events.py` for the NEWS half (independently recomputes
`NEWS_BLACKOUT` status per `NewsManager.mqh`/section 10 for every journal decision, directly
useful given every real journal record's `news_state` is currently always empty -- the live
EA never sets it, see `analysis/schema.py`'s docstring), and `analysis/performance_breakdown.py`
for the time-of-day, session, mode, and news OUTCOME breakdowns.

**Fixed, 2026-07-22 Codex review finding (third round):** this notebook's own text
previously described an Asia/London/New-York UTC-hour session convention that no longer
exists in the code below -- the "Time-of-day breakdown" cell only ever groups by the real,
DERIVED `hour_of_day` dimension (see that cell's own docstring for why no invented
session-bucket substitute is used there).

**Corrected, 2026-07-22 Codex review finding (fourth round): this notebook previously
stopped at hour-of-day and news-window-membership, and explicitly claimed it did NOT
perform a session/mode/news OUTCOME breakdown at all** -- despite
`performance_breakdown.py`'s `OPTIONAL_DIMENSIONS` already supporting `session_state`,
`intraday_mode`, `news_state`, and `in_news_blackout`. The "Session / mode / news OUTCOME
breakdown" section below now actually reports win rate/expectancy by all four, on
clearly-labelled SYNTHETIC data (per reproducibility rule 7) -- the breakdown LOGIC does not
need to wait for real broker-session data, only the real-data VALUES do (see the closing
cell for what's still needed to run this same breakdown for real).

**Uses clearly-labelled SYNTHETIC journal/news/trade fixtures.** Real-data run: PENDING.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.join_news_events import run as run_news_join

## News-blackout join

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_news_demo_"))

decision = {
    "signal_id": "sig-1",
    "timestamp_utc": "2026-07-21T14:05:30Z",
    "symbol": "XAUUSD",
    "market_family": "METAL",
    "intraday_mode": "SCALP",
    "regime": "REGIME_TRENDING_UP",
    "regime_confidence": 72.5,
    "direction": "BUY",
    "strategy": "TrendFollowingStrategy",
    "setup": "TrendlinePullback",
    "candlestick_pattern": None,
    "chart_pattern": None,
    "score": 68.0,
    "score_breakdown": {},
    "entry": 2350.55,
    "stop": 2345.10,
    "targets": [2361.45],
    "risk_percent": 0.3,
    "news_state": "",
    "session_state": "",
    "reasons_passed": [],
    "reasons_rejected": [],
    "ea_version": "1.01",
    "git_commit": "abc",
}
(tmp_dir / "decisions_20260721.jsonl").write_text(json.dumps(decision) + "\n", encoding="utf-8")

pd.DataFrame(
    [
        {
            "event_id": "e-nfp",
            "event_name": "NFP",
            "currency": "USD",
            "importance": 2,
            "scheduled_utc": "2026-07-21T14:10:00Z",  # 4m30s after the decision -- inside the window
        }
    ]
).to_csv(tmp_dir / "news.csv", index=False)

news_result = run_news_join(
    tmp_dir,
    tmp_dir / "news.csv",
    currency="USD",
    before_minutes=15,
    after_minutes=15,
    min_importance=2,
    repo_path=PROJECT_ROOT.parents[1],
)

print(f"n_decisions      = {news_result.n_decisions}")
print(f"n_in_blackout    = {news_result.n_in_blackout}")

assert news_result.n_in_blackout == 1
assert news_result.joined.iloc[0]["triggering_event_id"] == "e-nfp"

## Time-of-day breakdown (real, derived dimensions -- not an invented session bucket)

**Fixed, 2026-07-22 Codex review finding:** this cell previously defined a made-up
fixed-UTC-hour `session_for_hour` Asia/London/New-York bucketing function, computed inline
in the notebook rather than in a paired script -- and that bucketing was NOT sourced from
this project's actual session logic at all. `SessionManager.mqh` has **no fixed Asia/
London/New-York UTC-hour concept anywhere** -- it only ever computes session-time-remaining
from the BROKER'S OWN per-symbol session calendar (`SymbolInfoSessionTrade`, a live-MT5 API
this Python layer cannot call offline). Inventing a fixed-UTC-hour substitute was not
"porting the broker-session logic" as the master prompt requires; it was fabricating a
different, un-validated one.

This cell instead uses the real, paired `analysis/performance_breakdown.py` pipeline's
`hour_of_day`/`day_of_week` dimensions -- genuinely DERIVED from each decision's own
`entry_time` (no invented categorization). A true SESSION breakdown (Asia/London/NY-style)
would require either porting `SessionManager.mqh`'s real broker-session-table logic (needs a
real exported session table -- see `TASK-037_MT5_EXPORT_BRIDGE.md`) or consuming the
journal schema's own `session_state` field once the live EA populates it (see
`TASK-036_JOURNAL_PRODUCER_COMPLETION.md`) -- neither of which this notebook fabricates a
substitute for.

In [ ]:
import tempfile

from analysis.performance_breakdown import run as run_breakdown

# Trades entering at the same real UTC hour, three per hour, so the
# hour_of_day grouping below has genuine multi-trade statistics --
# hour_of_day is DERIVED from entry_time, not an invented session label.
synthetic_trades = pd.DataFrame(
    {
        "trade_id": [f"s{i}" for i in range(9)],
        "entry_time": (
            ["2026-07-21T02:00:00Z"] * 3
            + ["2026-07-21T09:00:00Z"] * 3
            + ["2026-07-22T15:00:00Z"] * 3
        ),
        "profit": [10.0, -5.0, 10.0, 10.0, 10.0, -5.0, -5.0, -5.0, 10.0],
    }
)
breakdown_dir = Path(tempfile.mkdtemp(prefix="themba_hourofday_demo_"))
trades_csv = breakdown_dir / "trades.csv"
synthetic_trades.to_csv(trades_csv, index=False)

performance_by_hour = run_breakdown(trades_csv, ["hour_of_day"])
print(performance_by_hour[["hour_of_day", "n_trades", "win_rate", "expectancy_dollars"]])

assert len(performance_by_hour) == 3
hour2_row = performance_by_hour[performance_by_hour["hour_of_day"] == 2].iloc[0]
assert abs(hour2_row["win_rate"] - (2.0 / 3.0)) < 1e-9  # 2 wins of 3
hour15_row = performance_by_hour[performance_by_hour["hour_of_day"] == 15].iloc[0]
assert abs(hour15_row["win_rate"] - (1.0 / 3.0)) < 1e-9  # 1 win of 3

## Session / mode / news OUTCOME breakdown (real composed pipeline, corrected 2026-07-22 Codex review finding, fifth round)

**Corrected, 2026-07-22 Codex review finding (fifth round): this section previously had two
defects the fourth-round version introduced.** Both are fixed below, not just re-labelled:

1. **The `session_state` bucket mapping was source-invalid.** `SessionManager.mqh`'s
   `SN_GetSessionMinutesRemaining` returns a continuous `remaining_ratio` measuring
   "fraction of today's remaining session time before the trading day's LAST configured
   session ends" -- it returns `1.0` *before the first session even opens* (not just while
   genuinely mid-session) and `false` (never a ratio) when there is no session today or the
   broker's session table is unreadable, under an explicit "exclude it, never default it"
   rule. Labelling `ratio >= 0.5` as `"OPEN"` therefore claimed something the function cannot
   actually establish (pre-open time and inter-session gaps are not "open"), and mapping
   every `false` to `"CLOSED"` turned a genuine data failure into a fabricated closed-session
   observation. The bucket names below describe exactly what the ratio measures -- remaining
   time, not an open/closed judgement -- with a genuine `SESSION_TIME_REMAINING_UNKNOWN`
   state for the `false` case: `ratio >= 0.5` -> `"SESSION_TIME_REMAINING_HIGH"`,
   `0.0 <= ratio < 0.5` -> `"SESSION_TIME_REMAINING_LOW"`, no session today / unreadable data
   -> `"SESSION_TIME_REMAINING_UNKNOWN"` (excluded from any HIGH/LOW judgement, never
   defaulted).

   **Made genuinely executable, not just documented, 2026-07-22 Codex review finding
   (sixth round): this mapping rule previously existed only as this markdown cell's own
   prose -- the code cell below hand-assigned the resulting bucket STRINGS directly,
   never derived them from a ratio, and never exercised the UNKNOWN case at all.
   `performance_breakdown.derive_session_state(remaining_ratio)` is that function now,
   real and unit-tested; the cell below feeds it actual ratios (including `None`, for
   the unreadable-session-table case) instead of hand-labelling the output.**
2. **The breakdown previously ran on a single, hand-assembled "already unified" DataFrame,
   never the actual composed pipeline.** Below instead runs the REAL chain: synthetic
   `decisions_*.jsonl` (with `order_id` set, `session_state`/`intraday_mode` populated) plus a
   real `news.csv` go through `join_news_events.py` (independently recomputes
   `in_news_blackout`/`triggering_event_id` from the news CSV -- NOT the journal's own
   self-reported `news_state`, since no numbered task has wired real news-state population
   into the live EA yet; `in_news_blackout` is the reliable signal), the result feeds
   `join_signal_to_outcome.py` together with a synthetic `trades.csv` to produce ONE unified
   per-position CSV, and THAT is what `performance_breakdown.py` actually groups -- the exact
   same functions a real MT5 export would run through (see `TASK-037_MT5_EXPORT_BRIDGE.md`
   items 9-10 for what real data still needs to exist to run this without the synthetic
   fixture).

`news_state` itself still has no defined real vocabulary (`NewsManager.mqh` exposes event
status/blackout/trigger-ID, not a decision-level label, and no numbered task populates the
journal's own `news_state` field yet) -- this section therefore breaks down by the
independently-computed `in_news_blackout` boolean, not by `news_state`, and does not invent a
vocabulary for the latter.


In [ ]:
import tempfile

from analysis.join_news_events import run as run_news_events_join
from analysis.join_signal_to_outcome import run as run_signal_to_outcome
from analysis.performance_breakdown import derive_session_state
from analysis.performance_breakdown import run as run_breakdown

# Hand-traceable synthetic decisions -- 9 positions spanning both the new
# source-faithful session_state buckets, the UNKNOWN case, and both
# intraday_mode values, with 3 genuinely falling inside a real
# news-blackout window (computed by join_news_events.py from news.csv
# below, not hand-labelled) so every breakdown has a clean,
# hand-computable answer AND is produced by the real composed pipeline,
# not pre-assembled.
composed_dir = Path(tempfile.mkdtemp(prefix="themba_composed_demo_"))

# **Added, 2026-07-22 Codex review finding (sixth round): this fixture
# previously hand-assigned the SESSION_TIME_REMAINING_HIGH/LOW bucket
# STRINGS directly, rather than deriving them from a
# SN_GetSessionMinutesRemaining-style ratio via derive_session_state --
# the mapping rule existed only as this notebook's own markdown prose.
# Ratios below are fed through the real function instead, and o9 (ratio
# None) exercises the UNKNOWN case, which this notebook previously never
# exercised at all.**
_DECISIONS = [
    # (order_id, hour, remaining_ratio,                intraday_mode, profit)
    ("o1", 0, 0.9, "SCALP", 10.0),
    ("o2", 1, 0.7, "SCALP", 10.0),
    ("o3", 2, 0.6, "DAY_TRADE", -5.0),
    ("o4", 3, 0.55, "DAY_TRADE", 10.0),
    ("o5", 4, 0.2, "SCALP", -5.0),  # in blackout
    ("o6", 5, 0.1, "SCALP", -5.0),  # in blackout
    ("o7", 6, 0.0, "DAY_TRADE", -5.0),  # in blackout
    ("o8", 7, 0.3, "DAY_TRADE", 10.0),
    ("o9", 8, None, "SCALP", 10.0),  # no session today / unreadable -> UNKNOWN
]
_BLACKOUT_ORDER_IDS = {"o5", "o6", "o7"}

decision_lines = []
for order_id, hour, remaining_ratio, intraday_mode, _profit in _DECISIONS:
    session_state = derive_session_state(remaining_ratio)
    decision_lines.append(
        json.dumps(
            {
                "signal_id": f"sig-{order_id}",
                "timestamp_utc": f"2026-07-21T{hour:02d}:00:00Z",
                "symbol": "XAUUSD",
                "market_family": "METAL",
                "intraday_mode": intraday_mode,
                "regime": "REGIME_TRENDING_UP",
                "regime_confidence": 70.0,
                "direction": "BUY",
                "strategy": "TrendFollowingStrategy",
                "setup": "TrendlinePullback",
                "candlestick_pattern": None,
                "chart_pattern": None,
                "score": 65.0,
                "score_breakdown": {},
                "entry": 2350.0,
                "stop": 2345.0,
                "targets": [2360.0],
                "risk_percent": 0.3,
                # No numbered task populates news_state on the live EA yet
                # (see markdown above) -- set here only so this synthetic
                # fixture is internally consistent with the independently
                # -computed in_news_blackout below (performance_breakdown.py
                # now rejects a news_state/in_news_blackout contradiction,
                # Codex review finding, 2026-07-22, fifth round); the real
                # dimension actually used for the breakdown is
                # in_news_blackout, not this self-reported field.
                "news_state": "BLACKOUT" if order_id in _BLACKOUT_ORDER_IDS else "CLEAR",
                "session_state": session_state,
                "reasons_passed": [],
                "reasons_rejected": [],
                "ea_version": "1.01",
                "git_commit": "abc",
                "order_id": order_id,
                "deal_id": None,  # no fill yet at decision time -- see join_signal_to_outcome.py's identity semantics
            }
        )
    )
(composed_dir / "decisions_20260721.jsonl").write_text(
    "\n".join(decision_lines) + "\n", encoding="utf-8"
)

# One real news event timed to fall inside o5/o6/o7's blackout window
# (default before/after = 15 minutes) and clearly outside everyone else's.
news_rows = [
    {
        "event_id": f"e-{order_id}",
        "event_name": "Synthetic high-importance release",
        "currency": "USD",
        "importance": 3,
        "scheduled_utc": f"2026-07-21T{hour:02d}:05:00Z",  # 5m after decision -- inside the 15m window
    }
    for order_id, hour, _ratio, _mode, _profit in _DECISIONS
    if order_id in _BLACKOUT_ORDER_IDS
]
pd.DataFrame(news_rows).to_csv(composed_dir / "news.csv", index=False)

news_result = run_news_events_join(
    composed_dir,
    composed_dir / "news.csv",
    currency="USD",
    before_minutes=15,
    after_minutes=15,
    min_importance=2,
    repo_path=PROJECT_ROOT.parents[1],
)
assert (
    set(news_result.joined.loc[news_result.joined["in_news_blackout"], "order_id"])
    == _BLACKOUT_ORDER_IDS
)

news_augmented_journal_csv = composed_dir / "journal_with_blackout.csv"
news_result.joined.to_csv(news_augmented_journal_csv, index=False)

trades_csv = composed_dir / "trades.csv"
pd.DataFrame(
    [
        {
            "trade_id": f"t-{order_id}",
            "order_id": order_id,
            "deal_id": f"d-{order_id}",
            "profit": profit,
        }
        for order_id, _hour, _ratio, _mode, profit in _DECISIONS
    ]
).to_csv(trades_csv, index=False)

unified_csv = composed_dir / "unified.csv"
joined_df, row_errors = run_signal_to_outcome(
    news_augmented_journal_csv,
    trades_csv,
    output_csv=unified_csv,
    repo_path=PROJECT_ROOT.parents[1],
)
assert row_errors == []
assert len(joined_df) == 9

performance_by_session_mode = run_breakdown(unified_csv, ["session_state", "intraday_mode"])
print(
    performance_by_session_mode[
        ["session_state", "intraday_mode", "n_trades", "win_rate", "expectancy_dollars"]
    ]
)

performance_by_blackout = run_breakdown(unified_csv, ["in_news_blackout"])
print(performance_by_blackout[["in_news_blackout", "n_trades", "win_rate", "expectancy_dollars"]])

# Hand-computed checks against the composed pipeline's real output (not a
# pre-assembled fixture): SESSION_TIME_REMAINING_HIGH/SCALP = o1,o2 both win
# -> win_rate 1.0; SESSION_TIME_REMAINING_LOW/DAY_TRADE = o7 (loss), o8 (win)
# -> win_rate 0.5; SESSION_TIME_REMAINING_UNKNOWN/SCALP = o9 alone, a win
# -> win_rate 1.0 (proves the UNKNOWN bucket is a genuine, independent
# group, not silently folded into HIGH/LOW or dropped); in_news_blackout=True
# = o5,o6,o7 all losses -> win_rate 0.0.
high_scalp = performance_by_session_mode[
    (performance_by_session_mode["session_state"] == "SESSION_TIME_REMAINING_HIGH")
    & (performance_by_session_mode["intraday_mode"] == "SCALP")
].iloc[0]
assert abs(high_scalp["win_rate"] - 1.0) < 1e-9

low_day_trade = performance_by_session_mode[
    (performance_by_session_mode["session_state"] == "SESSION_TIME_REMAINING_LOW")
    & (performance_by_session_mode["intraday_mode"] == "DAY_TRADE")
].iloc[0]
assert abs(low_day_trade["win_rate"] - 0.5) < 1e-9

unknown_scalp = performance_by_session_mode[
    (performance_by_session_mode["session_state"] == "SESSION_TIME_REMAINING_UNKNOWN")
    & (performance_by_session_mode["intraday_mode"] == "SCALP")
].iloc[0]
assert unknown_scalp["n_trades"] == 1
assert abs(unknown_scalp["win_rate"] - 1.0) < 1e-9

blackout_row = performance_by_blackout[performance_by_blackout["in_news_blackout"]].iloc[0]
assert abs(blackout_row["win_rate"] - 0.0) < 1e-9
assert blackout_row["n_trades"] == 3

## Real-data run: PENDING

The composed pipeline above (`join_news_events.py` -> `join_signal_to_outcome.py` ->
`performance_breakdown.py`) is now genuinely exercised end to end on synthetic-but-realistic
fixtures, proving the mechanism itself works, including the session-state vocabulary fix and
the independently-computed `in_news_blackout` dimension. What remains genuinely pending is
real data flowing through that SAME composed mechanism, which needs (see
`TASK-037_MT5_EXPORT_BRIDGE.md`, updated 2026-07-22 Codex review finding, fifth round, items
6-10 specifically):

1. `TASK-036`'s MQL5-side population of `intraday_mode`/`session_state` on real journal
   decisions, using the `SESSION_TIME_REMAINING_HIGH`/`LOW`/`UNKNOWN` vocabulary defined
   above (source-faithful to `SN_GetSessionMinutesRemaining`, not the old invalid
   `OPEN`/`CLOSING_SOON`/`CLOSED` mapping).
2. `TASK-037`'s real trade-history and news-calendar exports (Specification items 1-2) so
   `join_signal_to_outcome.py` and `join_news_events.py` have real input.
3. `TASK-037`'s session/news evidence export (Specification item 9) and the composed-run
   acceptance step (Specification item 10) -- both added 2026-07-22 Codex review finding,
   fifth round, specifically to own this real-data closure.

**Still unowned, named explicitly rather than silently missing (2026-07-22 Codex review
finding, fifth round):** no numbered task yet owns a `market_family`/`intraday_mode`
mode-router/classifier in the live EA itself -- `TASK-006_SESSION_MANAGER.md` explicitly
deferred that work, and `schema.py`'s own docstring already documents that
`ThembaAdaptiveIntradayEA.mq5` never sets either field today. A future task must be
registered for that classifier before `market_family`/`intraday_mode` can be genuinely
populated end to end; this notebook does not fabricate one.
